# EXP-001 — Lọc cứng: bỏ 25% ảnh khớp kém nhất (seed 1)
Lần trọng số mềm (`img-templ-sel-smile`, w 0.1–1) cho kết quả gần như trùng EXP-000 (|Δ| ≤ 0.77). Lần này ảnh có độ khớp LLM–detector thấp nhất bị **w=0**, còn lại **w=1** (loại đều 25% trong mỗi nhóm mức cười).

Dùng lại vector target, câu mẫu AU, `aus.csv` đã có trên Drive → **không chạy lại py-feat**. Chỉ: cài đặt → tính trọng số (vài phút) → train 5 epoch (~45 phút) → test → so sánh.

# Cấu hình

In [1]:
SEED = 1
DROP = 0.25                      # loại 25% ảnh khớp kém nhất (w=0), còn lại w=1
JOB_ID = f"exp001_au_drop{round(DROP*100)}_l14_seed{SEED}"
SOFT_JOB = f"exp001_au_l14_seed{SEED}"   # lần chạy trọng số mềm (đã xong)
BRANCH = "exp/001-au-detector-filtering"

# trọng số: cấu hình chọn ở test04, thêm lọc cứng --drop
POOL, PROTO, BALANCE = "img", "templ", "smile"
GROUPS_SEL = ["upper_lid_raise", "cheek_raise", "lip_corner_pull", "dimpler",
              "lip_stretch", "mouth_open", "eyes_closed"]   # Spearman > 0.1 ở test03
WEIGHTS_FILE = f"au_weights_{POOL}-{PROTO}-sel-{BALANCE}-drop{round(DROP*100)}.json"

INSTRUCTION = "Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image."
TRAIN_DATA = "/content/CAER-S_Face/train"
FACE_ZIP = "/content/drive/MyDrive/Datasets/CAERS_Face_train.zip"
TEST_ZIP = "/content/drive/MyDrive/Datasets/static_test_sets.zip"
PRE_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp001/full"     # precompute toàn bộ ảnh (dùng chung cho các seed)
OUT_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp001/train"    # checkpoint, log, kết quả test
T5_CACHE = "/root/.cache/huggingface/hub/models--google--flan-t5-xxl"
SCRIPT = "tools/precompute_au_weights.py"

import os, subprocess
def run(cmd):
    """Chạy lệnh shell, in output trực tiếp; lỗi thì dừng cell (khác với `!`)."""
    # bash + pipefail: lệnh có `| tee` vẫn báo lỗi nếu python lỗi
    p = subprocess.Popen("set -o pipefail; " + cmd, shell=True, executable="/bin/bash", text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})  # in log ngay, không đợi hết
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh lỗi (exit {p.returncode}) — xem output ở trên")

def hf_offline():
    # FlanT5 đã có trong cache thì không gọi HuggingFace nữa
    os.environ["HF_HUB_OFFLINE"] = "1" if os.path.isdir(T5_CACHE) else "0"

# Mount Drive & GPU

In [2]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(PRE_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
# BLIP-2 FlanT5-XXL cần ~24GB chỉ để nạp model → GPU 22–24GB (L4, A10) sẽ hết bộ nhớ
gpu_mem = int(subprocess.check_output("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits", shell=True).split()[0])
assert gpu_mem >= 35000, f"GPU chỉ có {gpu_mem} MiB — đổi runtime sang A100 (Runtime > Change runtime type)"

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


# Clone repo (nhánh EXP-001)

In [3]:
if not os.path.isdir("/content/Exp-CLIP"):
    !git clone -b $BRANCH https://github.com/huongng389/Exp-CLIP.git /content/Exp-CLIP
%cd /content/Exp-CLIP
!git checkout $BRANCH && git pull
!git log --oneline -1   # ghi lại commit vào research_log

Cloning into '/content/Exp-CLIP'...
remote: Enumerating objects: 374, done.
remote: Counting objects: 100% (374/374), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 374 (delta 277), reused 355 (delta 259), pack-reused 0 (from 0)
Receiving objects: 100% (374/374), 30.67 MiB | 43.56 MiB/s, done.
Resolving deltas: 100% (277/277), done.
/content/Exp-CLIP
Already on 'exp/001-au-detector-filtering'
Your branch is up to date with 'origin/exp/001-au-detector-filtering'.
Already up to date.
15d4084 (HEAD -> exp/001-au-detector-filtering, origin/exp/001-au-detector-filtering) EXP-001: add --drop hard filter to weight stage


# Giải nén dữ liệu (train + test)

In [4]:
!mkdir -p /content/CAER-S_Face /content/static_test_sets
!unzip -q -n "$FACE_ZIP" -d /content/CAER-S_Face
!unzip -q -n "$TEST_ZIP" -d /content/static_test_sets
!ls /content/CAER-S_Face/train
!ls /content/static_test_sets

Angry  Disgust	Fear  Happy  Neutral  Sad  Surprise
AffectNet7  AffectNet8	FERPlus  RAFDB


# Cài dependencies (venv310)\ntorch 2.0.1 + lavis: dùng cho target, proto, weight, train, test.

In [5]:
!sudo apt-get -qq update > /dev/null
!sudo apt-get -qq install -y python3.10 python3.10-venv python3.10-distutils > /dev/null
!python3.10 -m venv /content/venv310
!/content/venv310/bin/pip install -q --upgrade pip
!/content/venv310/bin/pip install -q --no-cache-dir torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!/content/venv310/bin/pip install -q --no-cache-dir transformers tqdm regex pyyaml pandas pydantic click timm einops
!/content/venv310/bin/pip install -q --no-cache-dir salesforce-lavis
# phải cài SAU lavis: lavis kéo numpy 2.x / opencv mới
!/content/venv310/bin/pip install -q --no-cache-dir "numpy<2" "opencv-python==4.10.0.84" "opencv-python-headless==4.10.0.84"
!/content/venv310/bin/pip install -q --no-cache-dir matplotlib scikit-learn
!/content/venv310/bin/python -c "import numpy, torch, lavis, sklearn; print('numpy', numpy.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 7.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 13.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Instal

# Kiểm tra dữ liệu precompute trên Drive

In [6]:
for f in ["protos.npz", "aus.csv", "targets"]:
    assert os.path.exists(f"{PRE_DIR}/{f}"), f"Thiếu {PRE_DIR}/{f} — chạy notebook EXP-001_train_seed1 trước"
!ls "$PRE_DIR/targets" | wc -l; wc -l "$PRE_DIR/aus.csv"

765
48939 /content/drive/MyDrive/ExpCLIP_output/exp001/full/aus.csv


# Trọng số lọc cứng
In ra số ảnh bị loại và tỉ lệ loại theo từng lớp CAER-S (chỉ để kiểm tra, không dùng nhãn).

In [7]:
run(f'/content/venv310/bin/python {SCRIPT} weight --out-dir {PRE_DIR} --pool {POOL} --proto {PROTO} '
    f'--groups {",".join(GROUPS_SEL)} --balance {BALANCE} --drop {DROP}')
!ls -lh "$PRE_DIR/$WEIGHTS_FILE"

[img-templ-sel-smile-drop25] 48938 targets | 48328 with AUs | 610 no face (w=1.0) | 0 not yet AU-detected (skipped)
agreement over 7 groups: mean 0.214 std 0.412 | weight mean 0.750 std 0.433 (map linear, balance smile, w_min 0.1)
drop 25%: 12082 images w=0 | agreement kept min -0.118 mean 0.401 | dropped per class: Angry 25%, Disgust 26%, Fear 26%, Happy 20%, Neutral 26%, Sad 27%, Surprise 25%
spearman per group: brow_raise -0.14, brow_lower +0.05, upper_lid_raise +0.20, cheek_raise +0.10, lid_tight -0.16, nose_wrinkle +0.02, lip_corner_pull +0.39, dimpler +0.14, lip_corner_depress -0.06, lip_stretch +0.23, lip_press +0.01, mouth_open +0.22, eyes_closed +0.32
-rw------- 1 root root 1.2M Oct 14 00:58 /content/drive/MyDrive/ExpCLIP_output/exp001/full/au_weights_img-templ-sel-smile-drop25.json


# Train seed 1 với trọng số lọc cứng
Đúng cấu hình EXP-000 (runner.sh), chỉ thêm `--sample-weights`. Checkpoint/log copy lên `OUT_DIR` sau mỗi epoch; bị ngắt thì chạy lại cell này (tự resume).

In [8]:
hf_offline()
resume = f"{OUT_DIR}/{JOB_ID}-resume.pth"
run(f'/content/venv310/bin/python train.py '
    f'--workers 8 --epochs 5 --batch-size 512 --batch-size-test-image 512 --batch-size-test-video 64 '
    f'--lr 1e-3 --weight-decay 1e-4 --momentum 0.9 --print-freq 10 --milestones 30 --seed {SEED} '
    f'--job-id {JOB_ID} --instruction "{INSTRUCTION}" --load-model CLIP_L14 '
    f'--train-data {TRAIN_DATA} --extra-checkpoint-dir {OUT_DIR} '
    f'--sample-weights {PRE_DIR}/{WEIGHTS_FILE}'
    + (f' --resume {resume}' if os.path.exists(resume) else ''))

Training date:  26-10-14 00:58
************************
workers = 8
epochs = 5
batch_size = 512
batch_size_test_image = 512
batch_size_test_video = 64
lr = 0.001
weight_decay = 0.0001
momentum = 0.9
print_freq = 10
milestones = [30]
seed = 1
job_id = exp001_au_drop25_l14_seed1
instruction = Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image.
load_model = CLIP_L14
extra_checkpoint_dir = /content/drive/MyDrive/ExpCLIP_output/exp001/train
train_data = /content/CAER-S_Face/train
sample_weights = /content/drive/MyDrive/ExpCLIP_output/exp001/full/au_weights_img-templ-sel-smile-drop25.json
resume = None
************************

  0%|                                               | 0.00/890M [00:00<?, ?iB/s]
  0%|                                       | 464k/890M [00:00<03:18, 4.69MiB/s]
  1%|▎                                     | 7.23M/890M [00:00<00:21, 43.5MiB/s]
  2%|▋                                     | 16.0M/

# Test zero-shot

In [9]:
!mkdir -p checkpoint log
if not os.path.exists(f"checkpoint/{JOB_ID}-model.pth"):
    !cp "$OUT_DIR/$JOB_ID-model.pth" checkpoint/
run(f'/content/venv310/bin/python test.py --load-model CLIP_L14 --job-id {JOB_ID} '
    f'--data-root /content/static_test_sets 2>&1 | tee log/{JOB_ID}-test.txt')
!cp log/$JOB_ID-test.txt "$OUT_DIR/"

************************
load_model = CLIP_L14
job_id = exp001_au_drop25_l14_seed1
data_root = /content/static_test_sets
************************
************************************************************************** Zero-shot Prompt Type:  Class Name
******************** Static FER Zero-shot Performance ********************
************************* RAFDB
UAR/WAR: 50.24/56.78
************************* AffectNet7
UAR/WAR: 42.54/42.54
************************* AffectNet8
UAR/WAR: 37.02/37.02
************************* FERPlus
UAR/WAR: 38.17/45.42
************************************************************************** Zero-shot Prompt Type:  An Expression of Name
******************** Static FER Zero-shot Performance ********************
************************* RAFDB
UAR/WAR: 56.60/64.08
************************* AffectNet7
UAR/WAR: 44.06/44.06
************************* AffectNet8
UAR/WAR: 38.33/38.33
************************* FERPlus
UAR/WAR: 39.12/52.62
*************************

# So sánh
Δ = lần này − mốc. Mốc chính: **EXP-000 seed1** (cùng seed, chỉ khác trọng số) và **EXP-001 soft seed1**. EXP-000 lệch giữa các seed ~0.5–1.8 điểm, nên Δ nhỏ hơn mức đó chưa đủ kết luận.

In [10]:
import re, pandas as pd
REF = pd.DataFrame([{"prompt": "Class Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 50.36, "WAR": 56.58}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 42.51, "WAR": 42.51}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 36.95, "WAR": 36.95}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 38.27, "WAR": 45.45}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 56.73, "WAR": 64.05}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.0, "WAR": 44.0}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 38.33, "WAR": 38.33}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 39.99, "WAR": 52.92}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 54.94, "WAR": 65.12}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 43.97, "WAR": 43.97}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 38.02, "WAR": 38.02}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 36.63, "WAR": 48.89}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 58.88, "WAR": 66.88}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.09, "WAR": 44.09}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 39.23, "WAR": 39.23}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 40.65, "WAR": 51.47}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 57.73, "WAR": 65.91}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.94, "WAR": 44.94}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 39.55, "WAR": 39.55}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 40.45, "WAR": 49.29}, {"prompt": "Class Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 50.23, "WAR": 54.56}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 41.47, "WAR": 41.47}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 35.76, "WAR": 35.76}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 38.58, "WAR": 45.82}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 55.34, "WAR": 62.0}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 43.79, "WAR": 43.79}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.4, "WAR": 38.4}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 38.89, "WAR": 51.62}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 55.64, "WAR": 64.78}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.39, "WAR": 44.39}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.08, "WAR": 38.08}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 37.95, "WAR": 49.17}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 58.07, "WAR": 65.32}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.17, "WAR": 44.17}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.84, "WAR": 38.84}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 41.55, "WAR": 53.41}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 57.19, "WAR": 64.6}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.97, "WAR": 44.97}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 39.45, "WAR": 39.45}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 42.17, "WAR": 49.46}, {"prompt": "Class Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}])

rows, prompt, dataset = [], None, None
for line in open(f"log/{JOB_ID}-test.txt"):
    if "Prompt Type:" in line:
        prompt = line.split("Prompt Type:")[1].strip()
    elif re.match(r"^\*+ (RAFDB|AffectNet7|AffectNet8|FERPlus)$", line.strip()):
        dataset = line.strip().split()[-1]
    elif line.startswith("UAR/WAR:"):
        uar, war = map(float, line.split(":")[1].strip().split("/"))
        rows.append(dict(prompt=prompt, dataset=dataset, UAR=uar, WAR=war))
res = pd.DataFrame(rows)
res.to_csv(f"{OUT_DIR}/{JOB_ID}-results.csv", index=False)
# lần trọng số mềm (EXP-001 seed1) để so trực tiếp
soft = pd.read_csv(f"{OUT_DIR}/{SOFT_JOB}-results.csv").assign(src="EXP-001 soft seed1")
# paper chỉ báo 1 số/dataset (Ensemble Five) → bỏ ở các prompt khác
REF = REF[(REF.src != "paper") | (REF.prompt == "Expression Ensemble Five")]
REF = pd.concat([REF, soft[REF.columns]])

table = res.set_index(["prompt", "dataset"])
for src in ["EXP-000 seed1", "EXP-001 soft seed1", "EXP-000 mean3", "paper"]:
    ref = REF[REF.src == src].set_index(["prompt", "dataset"])
    table[f"ΔUAR vs {src}"] = (table.UAR - ref.UAR).round(2)
table[f"ΔWAR vs EXP-000 seed1"] = (table.WAR - REF[REF.src == "EXP-000 seed1"].set_index(["prompt", "dataset"]).WAR).round(2)
pd.set_option("display.width", 250)
table

UAR    WAR  ΔUAR vs EXP-000 seed1  ΔUAR vs EXP-001 soft seed1  ΔUAR vs EXP-000 mean3  ΔUAR vs paper  ΔWAR vs EXP-000 seed1
prompt                                       dataset                                                                                                                                 
Class Name                                   RAFDB       50.24  56.78                  -0.12                        0.03                   0.01            NaN                   0.20
                                             AffectNet7  42.54  42.54                   0.03                       -0.06                   1.07            NaN                   0.03
                                             AffectNet8  37.02  37.02                   0.07                        0.07                   1.26            NaN                   0.07
                                             FERPlus     38.17  45.42                  -0.10                       -0.02                  -0.41            NaN                  -0.03
An Expression of Name                        RAFDB       56.60  64.08                  -0.13                        0.01                   1.26            NaN                   0.03
                                             AffectNet7  44.06  44.06                   0.06                        0.09                   0.27            NaN                   0.06
                                             AffectNet8  38.33  38.33                   0.00                        0.00                  -0.07            NaN                   0.00
                                             FERPlus     39.12  52.62                  -0.87                       -0.10                   0.23            NaN                  -0.30
A Photo of A Face with An Expression of Name RAFDB       54.75  65.19                  -0.19                       -0.13                  -0.89            NaN                   0.07
                                             AffectNet7  44.14  44.14                   0.17                        0.11                  -0.25            NaN                   0.17
                                             AffectNet8  38.07  38.08                   0.05                        0.02                  -0.01            NaN                   0.06
                                             FERPlus     36.73  49.29                   0.10                        0.03                  -1.22            NaN                   0.40
Expression Ensemble Five                     RAFDB       58.43  67.01                  -0.45                       -0.04                   0.36          -0.27                   0.13
                                             AffectNet7  44.11  44.11                   0.02                        0.05                  -0.06          -0.16                   0.02
                                             AffectNet8  39.05  39.05                  -0.18                        0.00                   0.21           0.61                  -0.18
                                             FERPlus     40.58  51.58                  -0.07                       -0.03                  -0.97          -7.70                   0.11
Expression Ensemble Ten                      RAFDB       57.72  66.04                  -0.01                       -0.12                   0.53            NaN                   0.13
                                             AffectNet7  44.91  44.91                  -0.03                       -0.03                  -0.06            NaN                  -0.03
                                             AffectNet8  39.52  39.52                  -0.03                       -0.08                   0.07            NaN                  -0.03
                                             FERPlus     40.57  49.62                   0.12                        0.02                  -1.60            NaN                   0.33

# Kiểm tra log train\nSo loss/accuracy từng epoch với EXP-000 (loss có trọng số nên giá trị tuyệt đối không so trực tiếp được, chỉ xem có hội tụ bình thường không).

In [11]:
!grep -E "Sample weights|The best accuracy|An epoch time" log/$JOB_ID-log.txt

Sample weights: 48938 / 48938 images found in /content/drive/MyDrive/ExpCLIP_output/exp001/full/au_weights_img-templ-sel-smile-drop25.json (others = 1), mean 0.753, min 0.000
The best accuracy: 1.342516541481018
An epoch time: 504.0861735343933s
The best accuracy: 5.437911510467529
An epoch time: 495.9686498641968s
The best accuracy: 12.64185905456543
An epoch time: 495.56236720085144s
The best accuracy: 21.036184310913086
An epoch time: 496.4676785469055s
The best accuracy: 27.685033798217773
An epoch time: 496.05239820480347s
